### Build and train a U-Net for building footprints

Use the paired NAIP imagery and reference land use/cover tiles prepared in Part 1 to train a **binary semantic segmentation** model. Each pixel is classified as non-building (`0`) or building (`1`); the tutorial uses reference code `5` for buildings. Confirm that code against your dataset's legend.

- Build a U-Net.
- Train and validate the model.
- Predict and visualize building footprint masks.

The homework at the end extends this workflow to a **multiclass land use/cover map**, retaining several reference categories instead of collapsing them into building/non-building.


### 0. Set up the environment (local Jupyter or optional Colab)

**Local workflow (default):** Open this notebook in JupyterLab, Jupyter Notebook, or VS Code using your Python environment. Run the setup cells first. They install only missing dependencies into the active kernel; set `INSTALL_MISSING_PACKAGES=False` if you manage packages yourself. Restart the kernel if installation requests it. The notebooks use `data/lab9` under the repository by default, whether launched from the repository root or the lab folder. Set `LOCAL_DATA_ROOT` to an absolute folder path if your data live elsewhere, and use the same path in both notebooks.

Run **Part 1 before Part 2** to prepare the paired training and validation tiles. Existing downloaded rasters are reused. Downloads and tile generation can take time and disk space.

**Optional Colab workflow:** The same cells run in Colab using `/content/lab9`. Google Drive is disabled by default. Set `USE_GOOGLE_DRIVE=True` in both notebooks if you want to mount Drive and share persistent data between their separate runtimes. Otherwise, transfer the prepared `dataset` folder between runtimes and retain your outputs before a runtime closes. No Colab import or Drive login is required locally.

Training uses CUDA when available and otherwise CPU. In Colab, optionally select a GPU runtime before running setup. For a platform-specific GPU build locally, install a matching PyTorch/torchvision pair for your hardware before running this notebook.


In [17]:
## If you don't have the modules installed, uncomment the following lines and run them to install the required modules.
#!pip install skimage  
#!pip install torchvision

In [18]:
import importlib.util
import subprocess
import sys

INSTALL_MISSING_PACKAGES = True  # Set False to manage your environment yourself.
packages = {'numpy': 'numpy', 'rasterio': 'rasterio', 'shapely': 'shapely', 'skimage': 'scikit-image', 'scipy': 'scipy', 'PIL': 'pillow', 'matplotlib': 'matplotlib', 'torch': 'torch', 'torchvision': 'torchvision'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING_PACKAGES:
        raise ImportError('Install these packages in this notebook kernel: ' + ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('Required packages are already installed.')


Required packages are already installed.


In [24]:
from pathlib import Path
import sys
import random
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
USE_GOOGLE_DRIVE = False  # Optional: enable only when running in Colab.
# Change this to an absolute path to use your own local data folder.
repository_dir = Path.cwd().resolve()
if repository_dir.name == 'lab9-image-segmentation':
    repository_dir = repository_dir.parent
LOCAL_DATA_ROOT = repository_dir / 'data' / 'lab9'

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/ai-urban-sustainability/lab9')
elif IN_COLAB:
    DATA_ROOT = Path('/content/lab9')
else:
    DATA_ROOT = Path(LOCAL_DATA_ROOT).expanduser().resolve()
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = DATA_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
random.seed(42)
np.random.seed(42)
print('Environment:', 'Colab' if IN_COLAB else 'local Python')
print('Data folder:', DATA_ROOT)
print('Output folder:', OUTPUT_DIR)
print('Python:', sys.version.split()[0])


Environment: local Python
Data folder: /Users/jianglix/Dropbox/xiaojiang/Teaching/Penn/AI-Urban-Sustainability/ai-urban-sustainability/data/lab9
Output folder: /Users/jianglix/Dropbox/xiaojiang/Teaching/Penn/AI-Urban-Sustainability/ai-urban-sustainability/data/lab9/outputs
Python: 3.14.7


In [25]:
import torch
import torchvision

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU selected; reduce epochs or batch size for a short demonstration.')


PyTorch: 2.14.1 | torchvision: 0.29.1
Device: cpu
CPU selected; reduce epochs or batch size for a short demonstration.


In [26]:
from torchvision.models import resnet50
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, sampler
from PIL import Image
import torch
# import matplotlib.pyplot as plt
import time
import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from torch import nn
from skimage import io, transform
from skimage.transform import rotate, AffineTransform, warp
import rasterio as rio
from torchvision.transforms import Resize, CenterCrop, Normalize
import torchvision.transforms as transforms
from scipy import ndimage


### 1. Build a U-Net

Define the network's computation in `forward(self, x)`. Keep PyTorch's inherited `nn.Module.__call__` so that `model(x)` invokes `forward` and preserves module hooks. Constructing `UNET(3, 2)` initializes the layers; it does not run a forward pass.

For this tutorial, three RGB input channels produce two output scores (logits) per pixel. `CrossEntropyLoss` consumes these logits directly, and `argmax(dim=1)` selects the predicted class. The current architecture expects tile dimensions divisible by 8; we use 512 × 512 tiles.


In [27]:
class UNET(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        #parameters: in_channels, out_channels, kernel_size, padding
        self.conv1 = self.contract_block(in_channels, 32, 7, 3)
        self.conv2 = self.contract_block(32, 64, 3, 1)
        self.conv3 = self.contract_block(64, 128, 3, 1)
        
        self.upconv3 = self.expand_block(128, 64, 3, 1)
        self.upconv2 = self.expand_block(64*2, 32, 3, 1)
        self.upconv1 = self.expand_block(32*2, out_channels, 3, 1)
        
    # nn.Module.__call__ invokes forward when we use model(x).
    def forward(self, x):
        # downsampling part
        conv1 = self.conv1(x)
        conv2 = self.conv2(conv1)
        conv3 = self.conv3(conv2)
        
        upconv3 = self.upconv3(conv3)
        upconv2 = self.upconv2(torch.cat([upconv3, conv2], 1))
        upconv1 = self.upconv1(torch.cat([upconv2, conv1], 1))
        
        return upconv1
    
    def contract_block(self, in_channels, out_channels, kernel_size, padding):
        contract = nn.Sequential(
            torch.nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding),
            torch.nn.BatchNorm2d(out_channels),
            torch.nn.ReLU(),
            torch.nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding),
            torch.nn.BatchNorm2d(out_channels),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
                                 )
        
        return contract
    
    def expand_block(self, in_channels, out_channels, kernel_size, padding):
        expand = nn.Sequential(torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=padding),
                            torch.nn.BatchNorm2d(out_channels),
                            torch.nn.ReLU(),
                            torch.nn.Conv2d(out_channels, out_channels, kernel_size, stride=1, padding=padding),
                            torch.nn.BatchNorm2d(out_channels),
                            torch.nn.ReLU(),
                            torch.nn.ConvTranspose2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1, output_padding=1) 
                            )
        return expand
        

In [28]:
unet = UNET(3,2)

unet

UNET(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (conv3)

### 2. Prepare imagery and building footprint labels

Resize categorical masks with **nearest-neighbor interpolation** to preserve their integer class codes. For the building tutorial only, convert reference code `5` to `1` and the other codes to `0`.

The tutorial below leaves augmentation disabled. If you enable geometric augmentation, sample each flip or rotation once and apply it to both the image and mask; independently calling random transforms on each can misalign them. Use nearest-neighbor interpolation for masks and treat any no-data pixels explicitly.


In [29]:
# this is used to augment the image
transform_aug = transforms.Compose([
     # transforms.ToPILImage(),
     # transforms.Resize((300, 300)),
     # transforms.CenterCrop((100, 100)),
     # transforms.RandomCrop((80, 80)),
     transforms.RandomHorizontalFlip(p=0.5),
     transforms.RandomRotation(degrees=(-90, 90)),
     transforms.RandomVerticalFlip(p=0.5)
     # transforms.ToTensor()
     # transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
     ])


class CanopyDataset(Dataset):
    def __init__(self, img_dir, msk_dir, pytorch=True, transforms=None):
        super().__init__()
        
        # Loop through the files in red folder and combine, into a dictionary, the other bands
        self.files = [self.combine_files(f, img_dir, msk_dir) for f in img_dir.iterdir() if not f.is_dir()]
        self.pytorch = pytorch
        self.transforms = transforms

        
    def combine_files(self, r_file: Path, img_dir, msk_dir):
        files = {'image': r_file, 
                 'mask': msk_dir/r_file.name.replace('naip_tiles', 'lu_masks')} #.replace('naip_tiles', 'lu_masks') is not necessary, just in case you have different name
        
        return files
        
    def __len__(self):
        return len(self.files)
    

    def open_as_array(self, idx, invert=False, include_nir=False, augment=False):
        img_pil = Image.open(self.files[idx]['image']).convert('RGB')
        # augment the image
        if augment:
            img_pil = self.transforms(img_pil)
            
        raw_rgb = np.asarray(img_pil).astype(float)
#         src_raw_rgb = rio.open(self.files[idx]['image'])
#         raw_rgb = src_raw_rgb.read()
#         src_raw_rgb.close()
        
        raw_rgb = transform.resize(raw_rgb, (512, 512, 3))
        if invert:
            raw_rgb = raw_rgb.transpose((2,0,1))
        
        # normalize
        # return (raw_rgb / np.iinfo(raw_rgb.dtype).max)
        return (raw_rgb / 255.0)
        
        
    def open_mask(self, idx, add_dims=False, augment=False):
        mask_pil = Image.open(self.files[idx]['mask'])
        # augment the image
        if augment:
            mask_pil = self.transforms(mask_pil)

        raw_mask = np.array(mask_pil).astype(float)
#         print('The filename is:=======', self.files[idx]['mask'])
#         src_raw_mask = rio.open(self.files[idx]['mask'])
#         raw_mask = src_raw_mask.read()
#         src_raw_mask.close()
        
        raw_mask = transform.resize(
            raw_mask, (512, 512), order=0, preserve_range=True, anti_aliasing=False
        )
        raw_mask = np.where(raw_mask == 5, 1, 0)  # Tutorial: reference code 5 is building; all other codes become non-building.
        
        return np.expand_dims(raw_mask, 0) if add_dims else raw_mask
        
        
    def __getitem__(self, idx):

        img_pil = Image.open(self.files[idx]['image']).convert('RGB')
        mask_pil = Image.open(self.files[idx]['mask'])
        # mask_y = np.array(mask_pil).astype(float)

        ## just the colorization of the raw image, not need to change the mask
        if self.transforms:
            img_pil = self.transforms(img_pil)
            mask_pil = self.transforms(mask_pil)

        image_x = np.asarray(img_pil).astype(float)
        mask_y = np.array(mask_pil).astype(float)

        image_x = transform.resize(image_x, (512, 512, 3))
        mask_y = transform.resize(
            mask_y, (512, 512), order=0, preserve_range=True, anti_aliasing=False
        )

        image_x = image_x.transpose((2,0,1))/255.0
        mask_y = np.where(mask_y == 5, 1, 0)  # Tutorial: reference code 5 is building; all other codes become non-building.

        # # 90 degree rotation
        # if np.random.rand()<0.5:
        #     angle = np.random.randint(4) * 90
        #     image_x = ndimage.rotate(image_x, angle,reshape=True)
        #     mask_y = ndimage.rotate(mask_y, angle, reshape=True)

        # # vertical flip
        # if np.random.rand()<0.5:
        #     image_x = np.flip(image_x, 0)
        #     mask_y = np.flip(mask_y, 0)
        
        # # horizonal flip
        # if np.random.rand()<0.5:
        #     image_x = np.flip(image_x, 1)
        #     mask_y = np.flip(mask_y, 1)

        ## add scale in future


        # x = torch.tensor(self.open_as_array(idx, \
        #                                     invert=self.pytorch, \
        #                                     include_nir=True, \
        #                                     augment=True), \
        #                                     dtype=torch.float32)
        # y = torch.tensor(self.open_mask(idx, add_dims=False, augment=True), \
        #                                 dtype=torch.torch.int64)
        
        # return x, y

        return torch.tensor(image_x, dtype=torch.float32), torch.tensor(mask_y, dtype=torch.int64)
    
    
    def open_as_pil(self, idx):
        arr = 256*self.open_as_array(idx)
        
        return Image.fromarray(arr.astype(np.uint8), 'RGB')
    
    def __repr__(self):
        s = 'Dataset class with {} files'.format(self.__len__())
        
        return s  
        

In [30]:
base_path = DATA_ROOT / 'dataset' / 'trainning'
validation_path = DATA_ROOT / 'dataset' / 'validation'
for directory in (base_path / 'imgs', base_path / 'labels',
                  validation_path / 'imgs', validation_path / 'labels'):
    if not directory.exists() or not any(directory.glob('*.tif')):
        raise FileNotFoundError(f'Run Part 1 first or set DATA_ROOT: missing tiles in {directory}')
train_ds = CanopyDataset(base_path / 'imgs', base_path / 'labels')
valid_ds = CanopyDataset(validation_path / 'imgs', validation_path / 'labels')
# Part 1 copies validation tiles; exclude those names from training.
validation_names = {item['image'].name for item in valid_ds.files}
train_ds.files = [item for item in train_ds.files
                  if item['image'].name not in validation_names]
if not train_ds.files:
    raise ValueError('No training tiles remain after excluding validation tiles.')
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=4, shuffle=False)
print('Training tiles:', len(train_ds), '| Validation tiles:', len(valid_ds))


Training tiles: 11655 | Validation tiles: 2913


### 3. Start training

In [31]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [32]:
device

device(type='cpu')

In [33]:

def train(model, train_dl, valid_dl, loss_fn, optimizer, acc_fn, epochs=1):
    start = time.time()
    # model.cuda()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = model.to(device)
    
    train_loss, valid_loss = [], []
    
    best_acc = 0.0
    
    for epoch in range(epochs):
        print('Epoch {}/{}'.format(epoch, epochs - 1))
        print('-' * 10)
        
        for phase in ['train', 'valid']:
            if phase == 'train':
                model.train(True)  # Set trainind mode = true
                dataloader = train_dl
            else:
                model.train(False)  # Set model to evaluate mode
                dataloader = valid_dl

            running_loss = 0.0
            running_acc = 0.0

            step = 0

            # iterate over data
            for x, y in dataloader:
                #x = x.cuda()
                #y = y.cuda()
                
                x, y = x.to(device), y.to(device)
                
                step += 1

                # forward pass
                if phase == 'train':
                    # zero the gradients
                    optimizer.zero_grad()
                    outputs = model(x)
                    loss = loss_fn(outputs, y)

                    # the backward pass frees the graph memory, so there is no 
                    # need for torch.no_grad in this training pass
                    loss.backward()
                    optimizer.step()
                    # scheduler.step()
                    
                else:
                    with torch.no_grad():
                        outputs = model(x)
                        loss = loss_fn(outputs, y.long())
                        
                # stats - whatever is the phase
                acc = acc_fn(outputs, y)
                
                running_acc  += acc*dataloader.batch_size
                running_loss += loss*dataloader.batch_size 
                
                # if step % 100 == 0:
                #     # clear_output(wait=True)
                #     print('Current step: {}  Loss: {}  Acc: {}  AllocMem (Mb): {}'.format(step, loss, acc, torch.cuda.memory_allocated()/1024/1024))
                #     # print(torch.cuda.memory_summary())

                if step % 100 == 0:
                    if device.type == "cuda":
                        mem = torch.cuda.memory_allocated() / 1024 / 1024
                        print(f"Step {step}: Loss={loss:.4f}, Acc={acc:.4f}, GPU Mem={mem:.1f} MB")
                    else:
                        print(f"Step {step}: Loss={loss:.4f}, Acc={acc:.4f}")


            
            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_acc / len(dataloader.dataset)
            
            # clear_output(wait=True)
            print('Epoch {}/{}'.format(epoch, epochs - 1))
            print('-' * 10)
            print('{} Loss: {:.4f} Acc: {}'.format(phase, epoch_loss, epoch_acc))
            print('-' * 10)

            train_loss.append(epoch_loss) if phase=='train' else valid_loss.append(epoch_loss)
    

    time_elapsed = time.time() - start
    print('Training complete in {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))    
    
    return train_loss, valid_loss    

# def batch_to_img(xb, idx):
#     img = np.array(xb[idx,0:3])
#     return img.transpose((1,2,0))
def batch_to_img(xb, idx):
    """Convert a tensor batch to image array."""
    img = np.array(xb[idx, 0:3].cpu())  # ensure it's on CPU for visualization
    return img.transpose((1, 2, 0))


# def predb_to_mask(predb, idx):
#     p = torch.functional.F.softmax(predb[idx], 0)
#     return p.argmax(0).cpu()
def predb_to_mask(predb, idx):
    """Convert model output to segmentation mask."""
    p = torch.nn.functional.softmax(predb[idx], dim=0)
    return p.argmax(0).cpu()


# def acc_metric(predb, yb):
#     return (predb.argmax(dim=1) == yb.cuda()).float().mean()
def acc_metric(predb, yb):
    """Compute accuracy on either CPU or GPU."""
    # both tensors are already on device
    return (predb.argmax(dim=1) == yb).float().mean()


In [34]:
unet = UNET(3,2).to(device)

In [ ]:
EPOCHS = 5  # Start with a short run; increase for your final experiment.
model_path = OUTPUT_DIR / 'unet_building.pth'
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(unet.parameters(), lr=0.001)
train_loss, valid_loss = train(unet, train_dl, valid_dl, loss_fn, opt,
                               acc_metric, epochs=EPOCHS)
torch.save(unet.state_dict(), model_path)
print('Saved checkpoint:', model_path)


Using device: cpu
Epoch 0/4
----------
Step 100: Loss=0.5352, Acc=0.7925
Step 200: Loss=0.4593, Acc=0.8024
Step 300: Loss=0.3859, Acc=0.8585
Step 400: Loss=0.3945, Acc=0.8204
Step 500: Loss=0.2882, Acc=0.8927
Step 600: Loss=0.2726, Acc=0.9054


### 4. Apply the trained model for building footprint segmentation


In [ ]:
def combine_files(r_file: Path, img_dir, msk_dir):
    files = {'image': r_file, 
             'mask': msk_dir/r_file.name.replace('naip_tiles', 'lu_masks')}
    
    print('r_file is:', r_file)
    print('msk_dir:', msk_dir)
    print('r_file.name is:', r_file.name.replace('naip_tiles', 'lu_masks'))
    print('The mas file is: -----', msk_dir/r_file.name.replace('naip_tiles', 'lu_masks'))
    return files
    

In [ ]:
img_dir = validation_path / 'imgs'
msk_dir = validation_path / 'labels'
img_dir, msk_dir


In [ ]:
# Loop through the files in red folder and combine, into a dictionary, the other bands
self_files = [combine_files(f, img_dir, msk_dir) for f in img_dir.iterdir() if not f.is_dir()]
self_files


Check the image and the label data

In [ ]:
from matplotlib import pyplot as plt

xb, yb = next(iter(train_dl))
print(xb.shape, yb.shape)

bs = 4
fig, ax = plt.subplots(bs, 2, figsize=(12, bs*5))
for i in range(bs):
    ax[i,0].imshow(batch_to_img(xb,i))
    ax[i,1].imshow(yb[i])
    


compare the model prediction and the reference data

In [ ]:
model = UNET(3,2)

model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model = model.to(device)
model.eval()

xb, yb = next(iter(valid_dl))

with torch.no_grad():
    predb = model(xb.to(device))

## batch size
bs = len(xb)
fig, ax = plt.subplots(bs, 3, figsize=(15, bs*5), squeeze=False)
for i in range(bs):
    ax[i,0].imshow(batch_to_img(xb,i))
    ax[i,1].imshow(yb[i])
    ax[i,2].imshow(predb_to_mask(predb, i))
    

### 5. Homework: multiclass land use/cover mapping with U-Net

**Objective:** Adapt the building footprint tutorial to predict one land use/cover category for every valid pixel in NAIP imagery. Your result must contain at least **three meaningful categories**, including buildings, rather than a binary building mask. Use the categories available in the supplied reference raster (for example, vegetation, buildings, water, and other surfaces if supported by its legend). Do not invent class names from pixel values: verify the source legend and document any regrouping. Imagery primarily reveals land cover; discuss limitations when interpreting functional land use.

#### Assignment

1. **Inspect and prepare the labels.** List the raw raster codes and their meanings, identify no-data values, and create a table mapping each selected source code to a contiguous training ID (`0, ..., C-1`). Report pixel counts per class. Keep image/mask grids aligned, including CRS, extent, and pixel size. Use nearest-neighbor resampling for labels. Explain how you handle excluded categories and no-data pixels.
2. **Create a spatial split.** Reserve separate geographic areas for training, validation, and testing before augmentation. Keep neighboring or overlapping tiles out of different splits to reduce spatial leakage. Show the split on a map and report tile counts. Use validation data to select settings; evaluate on the held-out test area only after those choices are fixed.
3. **Adapt and train U-Net.** Replace the binary mask conversion in every label-loading path with your multiclass mapping. Set `out_channels=C`, train with `CrossEntropyLoss`, and plot training and validation loss. Record the class mapping, input bands, normalization, split, random seed, learning rate, batch size, and epochs. Save the selected model checkpoint. If classes are imbalanced, try a documented strategy such as training-set class weights and compare its effect.
4. **Evaluate each category.** Report a confusion matrix, per-class intersection over union (IoU), and mean IoU on valid test pixels. Also report pixel accuracy, but explain why a dominant class can make it misleading. State how you handle classes absent from the test reference and predictions when averaging IoU. Show at least three test examples with RGB imagery, reference labels, and predictions using the same categorical colors and legend. Discuss errors at boundaries and between similar classes.
5. **Generate a map.** Run inference over the held-out area and assemble the predictions into a categorical, georeferenced GeoTIFF. Preserve the output grid's CRS and transform, assign a no-data code outside the class IDs, and include a class legend. If you resize tiles to 512 × 512 for the model, resize predictions back to the source grid using nearest-neighbor interpolation before writing them with the source transform. Describe your strategy for tile edges or overlapping predictions.

#### Hints

- Inspect codes across the reference tiles with `np.unique(raw_mask, return_counts=True)` and consult the source legend. The tutorial's `np.where(mask_y == 5, 1, 0)` discards all distinctions among non-building classes; **replace it**, rather than adding more binary masks.
- For `C` classes, images have shape `[N, 3, H, W]`, labels have shape `[N, H, W]` with dtype `torch.long`, and logits have shape `[N, C, H, W]`. Labels contain integer IDs, not RGB colors or one-hot vectors. Use RGB input as in the tutorial; adding NIR requires consistent changes to loading, normalization, and `in_channels`.
- Use the following pattern inside your dataset loader after resizing the raw mask. Define `source_codes` from the verified legend; the order defines your class IDs. Include every valid category you intend to predict. Explicitly ignore or regroup the rest, and report that choice.

```python
# Hint only: define source_codes from your reference legend before using this.
# source_codes = [...]  # C verified raster codes, in your chosen class order
IGNORE_INDEX = 255  # reserved value; choose another if this is a valid class ID
mask_y = np.full(raw_mask.shape, IGNORE_INDEX, dtype=np.int64)
for class_id, source_code in enumerate(source_codes):
    mask_y[raw_mask == source_code] = class_id
# Return torch.tensor(mask_y, dtype=torch.long) along with the image.

C = len(source_codes)
model = UNET(in_channels=3, out_channels=C).to(device)
loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
# Keep logits unchanged for the loss; do not apply softmax first.
logits = model(xb.to(device))  # PyTorch calls forward()
predicted_ids = logits.argmax(dim=1)
```

- Avoid batches with no valid labels: the mean loss is undefined when all pixels are ignored. Mask ignored pixels out of accuracy, confusion matrices, and IoU; the tutorial's `acc_metric` does not do this automatically.
- Use shared geometric transforms for each image/mask pair. Apply color or brightness changes only to imagery. Never interpolate categorical labels with bilinear interpolation.
- For evaluation and mapping, call `model.eval()` and run `model(x)` inside `torch.no_grad()`. `eval()` controls BatchNorm behavior; disabling gradients alone does not. Load the checkpoint on the intended device and instantiate the same `C` used during training.
- For each class, `IoU = TP / (TP + FP + FN)`. Exclude ignored pixels before counting. Use a discrete color palette with fixed class limits so that colors represent the same categories in every plot.
- Save the class-ID mapping alongside the checkpoint and GeoTIFF. Predicted training IDs differ from the original raster codes; either export the training IDs with their legend or map them back to source codes where that mapping is unambiguous.

#### Deliverables

Submit an executed notebook with your adapted dataset/model, reproducible training settings, loss curves, test metrics, and example maps; a categorical GeoTIFF of the held-out area with its legend; and a short report (1–2 pages) describing the class definitions, spatial split, results, main errors, and limitations. Include the saved checkpoint or a download link if it is too large to submit.

**Assessment:** label preparation and class mapping (25%), model adaptation and training (25%), spatial evaluation and error analysis (25%), and georeferenced map plus clear documentation (25%).
